# ✦ LILY WAN 2.2 — CLEAN v8

A clean rebuild: no nested patch wrappers, no Hugging Face Python downloader/Xet, no venv, no RIFE startup. Direct resumable model downloads; adaptive T4x2/P100 profile.


In [ ]:

import os, sys, subprocess, time, json, shutil, uuid, random, traceback, io
from pathlib import Path

ROOT = Path('/kaggle/working')
COMFY = ROOT / 'ComfyUI'
OUT = ROOT / 'lily_outputs'
OUT.mkdir(parents=True, exist_ok=True)

print('✦ Lily Wan 2.2 Studio — CLEAN v8')

# ---------- hardware ----------
try:
    raw = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True)
    GPU_NAMES = [x.strip() for x in raw.splitlines() if x.strip()]
except Exception:
    GPU_NAMES = []
print('Detected GPUs:', GPU_NAMES)
DUAL_T4 = len(GPU_NAMES) >= 2 and all('T4' in x.upper() for x in GPU_NAMES[:2])
P100 = len(GPU_NAMES) >= 1 and 'P100' in GPU_NAMES[0].upper()
PROFILE = 'DUAL_T4' if DUAL_T4 else ('P100' if P100 else 'SAFE_16GB')
print('✓ Hardware profile:', PROFILE)


def run(cmd, cwd=None, env=None, check=True, timeout=None):
    print('›', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env, check=check, text=True, timeout=timeout)

# ---------- minimal Python deps ----------
# Do NOT run pip --upgrade and do NOT let pip resolve a second CUDA/Torch stack.
missing = []
for mod, pkg in [('gradio','gradio'), ('requests','requests'), ('PIL','Pillow')]:
    try:
        __import__(mod)
    except Exception:
        missing.append(pkg)
if missing:
    run([sys.executable,'-m','pip','install','-q','--no-deps','--disable-pip-version-check',*missing])

import requests
import gradio as gr
from PIL import Image

# ---------- ComfyUI ----------
if not COMFY.exists():
    run(['git','clone','--depth','1','https://github.com/Comfy-Org/ComfyUI.git',str(COMFY)])
else:
    print('✓ Reusing existing ComfyUI folder')

req = COMFY / 'requirements.txt'
filtered = COMFY / 'requirements_lily_no_torch.txt'
skip = {'torch','torchvision','torchaudio'}
lines = []
for line in req.read_text().splitlines():
    s = line.strip()
    if not s or s.startswith('#'):
        lines.append(line)
        continue
    base = s.split(';',1)[0].strip().lower()
    name = base
    for sep in ('==','>=','<=','~=','>','<'):
        name = name.split(sep,1)[0].strip()
    if name in skip:
        print('✓ Reusing Kaggle system package:', name)
        continue
    lines.append(line)
filtered.write_text(chr(10).join(lines) + chr(10))
print('Installing only ComfyUI top-level requirements (NO dependency recursion)...')
run([sys.executable,'-m','pip','install','-q','--no-deps','--disable-pip-version-check','-r',str(filtered)], cwd=COMFY)
print('✓ ComfyUI requirements ready')

# ---------- optional MagCache ----------
MAG = COMFY / 'custom_nodes' / 'ComfyUI-MagCache'
MAGCACHE_WANTED = True
if not MAG.exists():
    try:
        run(['git','clone','--depth','1','https://github.com/Zehong-Ma/ComfyUI-MagCache.git',str(MAG)], timeout=90)
    except Exception as e:
        print('⚠ MagCache clone skipped:', e)
        MAGCACHE_WANTED = False
if MAG.exists():
    try:
        nodes_py = MAG / 'nodes.py'
        if nodes_py.exists():
            s = nodes_py.read_text()
            needle = '"wan2.1_vace_14B"], {"default":'
            replacement = '"wan2.1_vace_14B", "wan2.2_ti2v_5B"], {"default":'
            if needle in s and '"wan2.2_ti2v_5B"], {"default":' not in s:
                nodes_py.write_text(s.replace(needle, replacement, 1))
        run([sys.executable,'-m','pip','install','-q','--no-deps','--disable-pip-version-check','einops>=0.7.0','diffusers>=0.31.0'], check=False)
    except Exception as e:
        print('⚠ MagCache setup skipped:', e)

# ---------- model downloads: direct HTTP, no huggingface_hub/xet ----------
MODELS = [
    ('wan2.2_ti2v_5B_fp16.safetensors', COMFY/'models'/'diffusion_models',
     'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors'),
    ('umt5_xxl_fp8_e4m3fn_scaled.safetensors', COMFY/'models'/'text_encoders',
     'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
    ('wan2.2_vae.safetensors', COMFY/'models'/'vae',
     'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors'),
]

def attached(name):
    try:
        for p in Path('/kaggle/input').rglob(name):
            if p.is_file():
                return p
    except Exception:
        pass
    return None


def direct_download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 50_000_000:
        print('✓ model present:', dest.name)
        return dest
    a = attached(dest.name)
    if a:
        if dest.exists() or dest.is_symlink(): dest.unlink()
        dest.symlink_to(a)
        print('✓ using attached Kaggle model:', dest.name)
        return dest

    part = Path(str(dest) + '.part')
    done = part.stat().st_size if part.exists() else 0
    headers = {'Range': f'bytes={done}-'} if done else {}
    print(f'↓ {dest.name} — resume at {done/1e9:.2f} GB')
    with requests.get(url, stream=True, allow_redirects=True, headers=headers, timeout=(30,180)) as r:
        r.raise_for_status()
        mode = 'ab'
        if done and r.status_code == 200:
            done = 0
            mode = 'wb'
        total = r.headers.get('content-length')
        total = int(total) + done if total and r.status_code == 206 else (int(total) if total else None)
        last_print = done
        with open(part, mode) as f:
            current = done
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if not chunk: continue
                f.write(chunk)
                current += len(chunk)
                if current - last_print >= 256*1024*1024:
                    if total:
                        print(f'  {current/1e9:.2f}/{total/1e9:.2f} GB ({100*current/total:.0f}%)')
                    else:
                        print(f'  {current/1e9:.2f} GB')
                    last_print = current
    part.replace(dest)
    print('✓ downloaded:', dest.name)
    return dest

for name, folder, url in MODELS:
    direct_download(url, folder/name)

# ---------- launch ComfyUI ----------
URL = 'http://127.0.0.1:8188'
LOG = ROOT / 'comfyui.log'

def alive():
    try:
        return requests.get(URL + '/system_stats', timeout=2).ok
    except Exception:
        return False

if not alive():
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = '0'
    cmd = [sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch','--lowvram']
    with open(LOG,'w') as logf:
        PROC = subprocess.Popen(cmd, cwd=str(COMFY), env=env, stdout=logf, stderr=subprocess.STDOUT)
    print('Starting ComfyUI backend...')
    for i in range(180):
        if alive(): break
        time.sleep(1)
    else:
        tail = chr(10).join(LOG.read_text(errors='ignore').splitlines()[-100:]) if LOG.exists() else ''
        raise RuntimeError('ComfyUI failed to start. Last log lines:' + chr(10) + tail)
print('✓ ComfyUI backend ready')

obj = requests.get(URL + '/object_info', timeout=30).json()
required = ['UNETLoader','CLIPLoader','VAELoader','LoadImage','CLIPTextEncode','Wan22ImageToVideoLatent','ModelSamplingSD3','KSampler','VAEDecode','CreateVideo','SaveVideo']
missing_nodes = [x for x in required if x not in obj]
if missing_nodes:
    raise RuntimeError('Missing ComfyUI nodes: ' + ', '.join(missing_nodes))
MAGCACHE_READY = 'MagCache' in obj
if MAGCACHE_READY:
    try:
        choices = obj['MagCache']['input']['required']['model_type'][0]
        MAGCACHE_READY = 'wan2.2_ti2v_5B' in choices
    except Exception:
        pass
print('MagCache:', 'READY' if MAGCACHE_READY else 'baseline')

NEG = 'overexposed, oversaturated, static frame, blurry details, subtitles, text, watermark, worst quality, low quality, jpeg artifacts, malformed anatomy, deformed hands, extra fingers, fused fingers, duplicate limbs, frozen motion, cluttered background'
if PROFILE == 'DUAL_T4':
    PRESETS = {
        '⚡ Turbo': {'frames':61,'steps':12,'fps':12.0,'cfg':5.0},
        '✨ Normal': {'frames':81,'steps':16,'fps':16.0,'cfg':5.0},
        '👑 Max': {'frames':121,'steps':20,'fps':24.0,'cfg':5.0},
    }
    SIZES = {'Landscape 16:9':(1280,704),'Portrait 9:16':(704,1280)}
else:
    PRESETS = {
        '⚡ Turbo': {'frames':49,'steps':10,'fps':10.0,'cfg':5.0},
        '✨ Normal': {'frames':61,'steps':12,'fps':12.0,'cfg':5.0},
        '👑 Max': {'frames':81,'steps':16,'fps':16.0,'cfg':5.0},
    }
    SIZES = {'Landscape 16:9':(832,480),'Portrait 9:16':(480,832)}

def upload_image(img):
    buf = io.BytesIO(); img.convert('RGB').save(buf, format='PNG'); buf.seek(0)
    name = 'lily_' + uuid.uuid4().hex + '.png'
    r = requests.post(URL+'/upload/image', files={'image':(name,buf.getvalue(),'image/png')}, data={'type':'input','overwrite':'true'}, timeout=120)
    r.raise_for_status(); d=r.json(); sub=d.get('subfolder','')
    return f"{sub}/{d['name']}" if sub else d['name']

def workflow(image_name, prompt, negative, preset, orientation, seed, magcache):
    p=PRESETS[preset]; w,h=SIZES[orientation]
    wf={
      '1':{'class_type':'UNETLoader','inputs':{'unet_name':'wan2.2_ti2v_5B_fp16.safetensors','weight_dtype':'default'}},
      '2':{'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},
      '3':{'class_type':'VAELoader','inputs':{'vae_name':'wan2.2_vae.safetensors'}},
      '4':{'class_type':'LoadImage','inputs':{'image':image_name}},
      '5':{'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['2',0]}},
      '6':{'class_type':'CLIPTextEncode','inputs':{'text':negative,'clip':['2',0]}},
      '7':{'class_type':'Wan22ImageToVideoLatent','inputs':{'width':w,'height':h,'length':p['frames'],'batch_size':1,'vae':['3',0],'start_image':['4',0]}},
    }
    model='1'
    if magcache and MAGCACHE_READY:
        wf['8']={'class_type':'MagCache','inputs':{'model':['1',0],'model_type':'wan2.2_ti2v_5B','magcache_thresh':0.06,'retention_ratio':0.20,'magcache_K':2,'start_step':0,'end_step':-1}}
        model='8'
    wf['9']={'class_type':'ModelSamplingSD3','inputs':{'model':[model,0],'shift':8.0}}
    wf['10']={'class_type':'KSampler','inputs':{'model':['9',0],'positive':['5',0],'negative':['6',0],'latent_image':['7',0],'seed':int(seed),'steps':p['steps'],'cfg':p['cfg'],'sampler_name':'uni_pc','scheduler':'simple','denoise':1.0}}
    wf['11']={'class_type':'VAEDecode','inputs':{'samples':['10',0],'vae':['3',0]}}
    wf['12']={'class_type':'CreateVideo','inputs':{'images':['11',0],'fps':p['fps']}}
    wf['13']={'class_type':'SaveVideo','inputs':{'video':['12',0],'filename_prefix':'video/LILY_WAN22','format':'auto','codec':'auto'}}
    return wf

def wait_prompt(pid, timeout=5400):
    t=time.time()
    while time.time()-t < timeout:
        r=requests.get(URL+f'/history/{pid}',timeout=30)
        if r.ok:
            h=r.json()
            if pid in h:
                item=h[pid]; st=item.get('status',{})
                if st.get('status_str')=='error':
                    raise RuntimeError(json.dumps(st.get('messages',[])[-5:],indent=2)[:5000])
                if st.get('completed',False): return
        time.sleep(2)
    raise TimeoutError('Generation timed out')

def newest(after):
    files=[]
    for ext in ('*.mp4','*.webm','*.mov','*.mkv'):
        files += list((COMFY/'output').rglob(ext))
    files=[p for p in files if p.stat().st_mtime >= after-2]
    if not files: raise FileNotFoundError('Generation finished but no video file was found')
    return max(files,key=lambda p:p.stat().st_mtime)

def finish_video(src,dst):
    run(['ffmpeg','-y','-i',str(src),'-vf','fps=24','-an','-c:v','libx264','-preset','veryfast','-crf','19','-pix_fmt','yuv420p','-movflags','+faststart',str(dst)])

def generate(image,prompt,negative,preset,orientation,seed,randomize,magcache):
    if image is None: raise gr.Error('Add an image first.')
    if not (prompt or '').strip(): raise gr.Error('Write a motion prompt.')
    if randomize: seed=random.randint(0,2**63-1)
    seed=int(seed)
    try:
        imgname=upload_image(image)
        wf=workflow(imgname,prompt.strip(),negative or NEG,preset,orientation,seed,bool(magcache))
        start=time.time()
        r=requests.post(URL+'/prompt',json={'prompt':wf},timeout=60)
        if not r.ok: raise RuntimeError(f'ComfyUI rejected workflow: {r.status_code} {r.text[:2500]}')
        pid=r.json()['prompt_id']
        wait_prompt(pid)
        raw=newest(start)
        final=OUT/f'lily_{seed}_{uuid.uuid4().hex[:6]}.mp4'
        finish_video(raw,final)
        mins=(time.time()-start)/60
        return str(final), f'✅ Done • {mins:.1f} min • {PROFILE} • seed {seed}', seed
    except Exception as e:
        tail=''
        try: tail=chr(10).join(LOG.read_text(errors='ignore').splitlines()[-50:])
        except Exception: pass
        print(traceback.format_exc())
        raise gr.Error(f'{type(e).__name__}: {e}' + chr(10) + chr(10) + tail[-4000:])

css='.gradio-container{max-width:920px!important;margin:auto!important;}'
with gr.Blocks(css=css,title='Lily Wan 2.2 Studio') as demo:
    gr.Markdown(f'# ✦ Lily Wan 2.2 Studio\n**Hardware:** {PROFILE} • **Wan 2.2 TI2V-5B**')
    with gr.Row():
        with gr.Column():
            image=gr.Image(label='Starting image',type='pil',sources=['upload','clipboard'])
            prompt=gr.Textbox(label='Motion prompt',lines=5)
            negative=gr.Textbox(label='Negative prompt',value=NEG,lines=2)
        with gr.Column():
            preset=gr.Radio(list(PRESETS),value='⚡ Turbo',label='Quality / speed')
            orientation=gr.Radio(list(SIZES),value='Portrait 9:16',label='Format')
            magcache=gr.Checkbox(value=MAGCACHE_READY,label='MagCache acceleration')
            seed=gr.Number(value=42,precision=0,label='Seed')
            randomize=gr.Checkbox(value=True,label='Randomize seed')
            go=gr.Button('✦ GENERATE VIDEO',variant='primary',size='lg')
    out=gr.Video(label='Result',autoplay=True)
    status=gr.Textbox(label='Status',interactive=False)
    used=gr.Number(label='Used seed',precision=0,interactive=False)
    go.click(generate,[image,prompt,negative,preset,orientation,seed,randomize,magcache],[out,status,used],concurrency_limit=1)
    demo.queue(max_size=10)
    print('✓ SETUP COMPLETE — opening Gradio share link')
    demo.launch(share=True,server_name='0.0.0.0',show_error=True,allowed_paths=[str(OUT),str(COMFY/'output')])
